In [ ]:
# Load CARAVAN data and calculate Knoben climate indices
# Assumes CARAVAN folder 'attributes' is located as a subfolder of the directory this notebook is inlatin h

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Find files - what we need is in the HydroAtlas CSV files
files = glob.glob('../attributes/*/attributes_hydroatlas*.csv')

In [ ]:
# Open files, extract columns of interest (pre, tmp, pet) and merge
columns = ['gauge_id', 
           'tmp_dc_s01','tmp_dc_s02','tmp_dc_s03','tmp_dc_s04',
           'tmp_dc_s05','tmp_dc_s06','tmp_dc_s07','tmp_dc_s08',
           'tmp_dc_s09','tmp_dc_s10','tmp_dc_s11','tmp_dc_s12',
           'pre_mm_s01','pre_mm_s02','pre_mm_s03','pre_mm_s04',
           'pre_mm_s05','pre_mm_s06','pre_mm_s07','pre_mm_s08',
           'pre_mm_s09','pre_mm_s10','pre_mm_s11','pre_mm_s12',
           'pet_mm_s01','pet_mm_s02','pet_mm_s03','pet_mm_s04',
           'pet_mm_s05','pet_mm_s06','pet_mm_s07','pet_mm_s08',
           'pet_mm_s09','pet_mm_s10','pet_mm_s11','pet_mm_s12']
caravan = pd.concat([pd.read_csv(file, usecols=columns) for file in files],
                    ignore_index=True)
caravan = caravan.set_index('gauge_id')

In [ ]:
# filter by size and length of data series



In [ ]:
def calculate_kci(df, T0=0):
    
    '''Calculates climate indices as in Eq. 1-4 of Knoben et al., 2019 '''
    
    # In: 
    # - df: pandas dataframe with columns pet_mm_s[01-12] and pre_mm_s[01-12]
    # - T0: optional (default = 0), threshold temperature for snowfall
    # Out:
    # - out: pandas dataframe with three index values
    
    # Prepare output
    out = pd.DataFrame(index=caravan.index,columns=['im','imr','fs'])
    
    # Loop over rows in input
    # Note: this can probably be streamlined with a lambda function or smth
    for ix,row in df.iterrows():
        
        # Empty arrays
        mi = np.zeros(12)
        fm = np.zeros(12)
        pp = np.zeros(12)
        
        # Loop over months and calculate MI (eq. 1) and if snow exists (eq. 4) 
        for s in range(1,13):
            month = f'{s:02}'

            # Get monthly values
            p = row[f'pre_mm_s{month}']
            e = row[f'pet_mm_s{month}']
            t = row[f'tmp_dc_s{month}'] / 10

            # Calculate MI (eq. 1)
            if p > e:
                mi[s-1] = 1-e/p
            elif p == e:
                mi[s-1] = 0
            else:
                mi[s-1] = p/e-1

            # Calculate monthly snow (top part of eq. 4)
            pp[s-1] = p # Needed for later summing
            if t < T0:
                fm[s-1] = p

        # Calculate Im (eq. 2), Imr (eq. 3) and fs (eq. 4)
        im = mi.mean()
        imr = mi.max()-mi.min()
        fs = fm.sum()/pp.sum()
        
        # Store in out
        out.at[row.name,'im'] = im
        out.at[row.name,'imr'] = imr
        out.at[row.name,'fs'] = fs
    
    return out

In [ ]:
kci = calculate_kci(caravan)

In [ ]:
# Define the colorscheme
kci['r'] = (-1*kci['im']+1)/2
kci['g'] = kci['imr']/2
kci['b'] = kci['fs']

In [ ]:
kci.to_csv('caravan_climate_indices.csv')

In [ ]:
# Plot the results
colors = np.array([kci['r'].values,kci['g'].values,kci['b'].values]).T
plt.rcParams.update({'font.size': 18})

lbl_im  = '$I_m$'
lbl_imr = '$I_{m,r}$'
lbl_fs  = '$f_s$'
rng_im  = [-1,1]
rng_imr = [0,2]
rng_fs  = [0,1]

fig,ax = plt.figure(figsize=(20,20))
#fig = plt.figure(figsize=(20,20))
fig.suptitle('CARAVAN catchments in climate index space')

# All 3
ax[0,0] = fig.add_subplot(221,projection='3d')
ax[0,0].scatter(kci['im'], kci['imr'], kci['fs'], c=colors)
ax[0,0].set_xlim(rng_im)
ax[0,0].set_ylim(rng_imr)
ax[0,0].set_zlim(rng_fs)
ax[0,0].set_xlabel(lbl_im)
ax[0,0].set_ylabel(lbl_imr)
ax[0,0].set_zlabel(lbl_fs)
ax[0,0].view_init(elev=30., azim=-135)

# Im vs Imr
ax[0,1] = fig.add_subplot(222)
ax[0,1].scatter(kci['im'],kci['imr'],c=colors)
#ax[0,1].axis('off')
ax[0,1].set_xlabel(lbl_im)
ax[0,1].set_ylabel(lbl_imr)
ax[0,1].set_xlim(rng_im)
ax[0,1].set_ylim(rng_imr)

# Imr vs fs
ax[1,0] = fig.add_subplot(223)
ax[1,0].scatter(kci['imr'],kci['fs'],c=colors)
ax[1,0].set_xlabel(lbl_imr)
ax[1,0].set_ylabel(lbl_fs)
ax[1,0].set_xlim(rng_imr)
ax[1,0].set_ylim(rng_fs)

# Im vs fs
ax[1,1] = fig.add_subplot(224)
ax[1,1].scatter(kci['im'],kci['fs'],c=colors)
ax[1,1].set_xlabel(lbl_im)
ax[1,1].set_ylabel(lbl_fs)
ax[1,1].set_xlim(rng_im)
ax[1,1].set_ylim(rng_fs)

plt.tight_layout()
plt.savefig('caravan_catchments_in_climate_index_space.png',dpi=300,bbox_inches='tight')

In [ ]:
# cluster the catchments

from sklearn.cluster import KMeans


In [ ]:
data = kci
data.head()

In [ ]:
X = data.iloc[:,0:3].values
print(data.iloc[:,0:3].values)

In [ ]:
wcss = []

for i in range(1,11):
    k_means = KMeans(n_clusters=i,init='k-means++', random_state=42)
    k_means.fit(X)
    wcss.append(k_means.inertia_)
#plot elbow curve
plt.plot(np.arange(1,11),wcss)
plt.xlabel('Clusters')
plt.ylabel('SSE')
plt.show()

In [ ]:
k_means_optimum = KMeans(n_clusters = 10, init = 'k-means++',  random_state=8)
y = k_means_optimum.fit_predict(X)
print(y)

In [ ]:
data['cluster'] = y  

In [ ]:
data1 = data[data.cluster==0]
data2 = data[data.cluster==1]
data3 = data[data.cluster==2]
data4 = data[data.cluster==3]
data5 = data[data.cluster==4]
data6 = data[data.cluster==5]
data7 = data[data.cluster==6]
data8 = data[data.cluster==7]
data9 = data[data.cluster==8]
data10 = data[data.cluster==9]

In [ ]:

#fig = plt.
fig = plt.figure(figsize=(20,20)) #subplots(2,2,figsize=(20,20))
#fig = plt.figure(figsize=(20,20))
fig.suptitle('K-means clustered CARAVAN catchments in climate index space')

ax[0,0] = fig.add_subplot(221,projection='3d')

#ax[0,0].set_axis_off()
#ax[0,0] = plt.axes(projection='3d')
#ax = ax[0,0].add_subplot(projection='3d')

#xline = np.linspace(0, 1, 1000)
#yline = np.linspace(0, 1, 1000)
#zline = np.linspace(0, 1, 1000)
#ax[0,0].plot3D(xline, yline, zline, 'black')
# Data for three-dimensional scattered points

ax[0,0].scatter3D(data1.im, data1.imr, data1.fs, c='red', label = 'Cluster 1')
ax[0,0].scatter3D(data2.im, data2.imr, data2.fs, c='green', label = 'Cluster 2')
ax[0,0].scatter3D(data3.im, data3.imr, data3.fs, c='blue', label = 'Cluster 3')
ax[0,0].scatter3D(data4.im, data4.imr, data4.fs, c='yellow', label = 'Cluster 4')
ax[0,0].scatter3D(data5.im, data5.imr, data5.fs, c='cyan', label = 'Cluster 5')
ax[0,0].scatter3D(data6.im, data6.imr, data6.fs, c='magenta', label = 'Cluster 6')
ax[0,0].scatter3D(data7.im, data7.imr, data7.fs, c='purple', label = 'Cluster 7')
ax[0,0].scatter3D(data8.im, data8.imr, data8.fs, c='pink', label = 'Cluster 8')
ax[0,0].scatter3D(data9.im, data9.imr, data9.fs, c='olive', label = 'Cluster 9')
ax[0,0].scatter3D(data10.im, data10.imr, data10.fs, c='lightblue', label = 'Cluster 10')
ax[0,0].scatter3D(k_means_optimum.cluster_centers_[:,0],k_means_optimum.cluster_centers_[:,1], k_means_optimum.cluster_centers_[:,2], color = 'indigo', s = 200)
for i in range(0,10):
    print(i)
    print(k_means_optimum.cluster_centers_[i,0])
    print(k_means_optimum.cluster_centers_[i,1])
    print(k_means_optimum.cluster_centers_[i,2])
    print(len(data[data.cluster==i]))


ax[0,0].set_xlabel('im')
ax[0,0].set_ylabel('imr')
ax[0,0].set_zlabel('fs')
#

# Im vs Imr
ax[0,1] = fig.add_subplot(222)
ax[0,1].scatter(data1.im, data1.imr,  c='red', label = 'Cluster 1')
ax[0,1].scatter(data2.im, data2.imr,  c='green', label = 'Cluster 2')
ax[0,1].scatter(data3.im, data3.imr,  c='blue', label = 'Cluster 3')
ax[0,1].scatter(data4.im, data4.imr, c='yellow', label = 'Cluster 4')
ax[0,1].scatter(data5.im, data5.imr,  c='cyan', label = 'Cluster 5')
ax[0,1].scatter(data6.im, data6.imr,  c='magenta', label = 'Cluster 6')
ax[0,1].scatter(data7.im, data7.imr,  c='purple', label = 'Cluster 7')
ax[0,1].scatter(data8.im, data8.imr,  c='pink', label = 'Cluster 8')
ax[0,1].scatter(data9.im, data9.imr,  c='olive', label = 'Cluster 9')
ax[0,1].scatter(data10.im, data10.imr,  c='lightblue', label = 'Cluster 10')
ax[0,1].scatter(k_means_optimum.cluster_centers_[:,0],k_means_optimum.cluster_centers_[:,1], color = 'indigo', s = 200)

ax[0,1].set_xlabel(lbl_im)
ax[0,1].set_ylabel(lbl_imr)
#ax[0,1].get_xaxis().set_visible(False)
#ax[0,1].spines['right'].set_visible(False)
#ax[0,1].set_xticks([-1,0,1])
ax[0,1].set_xlim(rng_im)
ax[0,1].set_ylim(rng_imr)
#ax[0,1].axis('off')

# Imr vs fs
ax[1,0] = fig.add_subplot(223)
ax[1,0].scatter(data1.imr, data1.fs, c='red', label = 'Cluster 1')
ax[1,0].scatter(data2.imr, data2.fs, c='green', label = 'Cluster 2')
ax[1,0].scatter(data3.imr, data3.fs, c='blue', label = 'Cluster 3')
ax[1,0].scatter(data4.imr, data4.fs, c='yellow', label = 'Cluster 4')
ax[1,0].scatter(data5.imr, data5.fs, c='cyan', label = 'Cluster 5')
ax[1,0].scatter(data6.imr, data6.fs, c='magenta', label = 'Cluster 6')
ax[1,0].scatter(data7.imr, data7.fs, c='purple', label = 'Cluster 7')
ax[1,0].scatter(data8.imr, data8.fs, c='pink', label = 'Cluster 8')
ax[1,0].scatter(data9.imr, data9.fs, c='olive', label = 'Cluster 9')
ax[1,0].scatter(data10.imr, data10.fs, c='lightblue', label = 'Cluster 10')
ax[1,0].scatter(k_means_optimum.cluster_centers_[:,1],k_means_optimum.cluster_centers_[:,2], color = 'indigo', s = 200)

ax[1,0].set_xlabel(lbl_imr)
ax[1,0].set_ylabel(lbl_fs)
ax[1,0].set_xlim(rng_imr)
ax[1,0].set_ylim(rng_fs)

# Im vs fs
ax[1,1] = fig.add_subplot(224)
ax[1,1].scatter(data1.im,  data1.fs, c='red', label = 'Cluster 1')
ax[1,1].scatter(data2.im,  data2.fs, c='green', label = 'Cluster 2')
ax[1,1].scatter(data3.im,  data3.fs, c='blue', label = 'Cluster 3')
ax[1,1].scatter(data4.im,  data4.fs, c='yellow', label = 'Cluster 4')
ax[1,1].scatter(data5.im,  data5.fs, c='cyan', label = 'Cluster 5')
ax[1,1].scatter(data6.im,  data6.fs, c='magenta', label = 'Cluster 6')
ax[1,1].scatter(data7.im,  data7.fs, c='purple', label = 'Cluster 7')
ax[1,1].scatter(data8.im,  data8.fs, c='pink', label = 'Cluster 8')
ax[1,1].scatter(data9.im,  data9.fs, c='olive', label = 'Cluster 9')
ax[1,1].scatter(data10.im, data10.fs, c='lightblue', label = 'Cluster 10')
ax[1,1].scatter(k_means_optimum.cluster_centers_[:,0],k_means_optimum.cluster_centers_[:,2], color = 'indigo', s = 200)

ax[1,1].set_xlabel(lbl_im)
ax[1,1].set_ylabel(lbl_fs)
ax[1,1].set_xlim(rng_im)
#ax[1,1].axis('off')
ax[1,1].set_ylim(rng_fs)


#plt.zlabel("fs")
#plt.legend()
#plt.title("Kmeans")
#plt.show()
plt.tight_layout()

In [ ]:
# use euclidian metric to find catchment closest to cluster centroid
import math
# for loop clusters
im_c = k_means_optimum.cluster_centers_[5,0]
imr_c = k_means_optimum.cluster_centers_[5,1]
fs_c = k_means_optimum.cluster_centers_[5,2]

data6["euklid"] = np.sqrt((data.im.astype(float)-im_c)**2+(data.imr.astype(float)-imr_c)**2+(data.fs.astype(float)-fs_c)**2)
#math.sqrt((data1.im-im_c)**2+(data1["imr"]-imr_c)**2+(data1["fs"]-fs_c)**2)


In [ ]:
# use euclidian metric to find catchment closest to cluster centroid
import math
# for loop clusters
im_c = k_means_optimum.cluster_centers_[0,0]
imr_c = k_means_optimum.cluster_centers_[0,1]
fs_c = k_means_optimum.cluster_centers_[0,2]

#data1['euklid'] = 

for i in range(0,10):
    euklid = pd.Series([])
    for j in range(0, len(data[data.cluster==i])):
        im_c = k_means_optimum.cluster_centers_[i,0]
        imr_c = k_means_optimum.cluster_centers_[i,1]
        fs_c = k_means_optimum.cluster_centers_[i,2]
        #p = np.array([im_c,imr_c,fs_c])
        #q = np.array([data.im[data.cluster==i][j],data.imr[data.cluster==i][j],data.fs[data.cluster==i][j]])
        
        # calculate distance to centroid for current data point
        euklid[j] = math.sqrt((data.im[data.cluster==i][j]-im_c)**2+(data.imr[data.cluster==i][j]-imr_c)**2+(data.fs[data.cluster==i][j]-fs_c)**2)  
        
        # check if distance is smaller than current closest
        #data['euclid']= np.where(data['cluster']==i,euklid[j],data['euclid'])

        # save current catchment name    
    #data['euclid'] = np.where(data[data['cluster']==i],euklid,'In Previous')
    #data['euclid'][data['cluster']==i]= np.where(data[data['cluster']==i],euklid,data['euclid'][data['cluster']==i])
    df2 = pd.concat([data[data['cluster'==i]].reset_index(drop=True),pd.DataFrame([euklid]).reset_index(drop=True)],axis=1)
    #data.assign(euclid=euklid)

    # print name for each catchment     




In [ ]:
del df2